<a href="https://colab.research.google.com/github/paulmunozpauta/Curso_Introduccion_Ciencia_Datos/blob/main/Notebooks/M02_03_Limpieza_preprocesamiento_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="center">
  <img src="https://github.com/paulmunozpauta/Curso_Introduccion_Ciencia_Datos/raw/main/Static/Imgs/UdeC_color_horizontal.jpg" width="500">
</p>

<p align="center"><b style="font-size:28px;">Facultad de Ingeniería Agrícola</b></p>
<p align="center"><b style="font-size:28px;">Introducción a la Ciencia de Datos</b></p>
<hr>
<p align="center"><b style="font-size:28px;">Contacto</b></p>
<p align="center">
  paulmunoz@udec.cl<br>
  https://paulmunoz.com
</p>

# Introducción a la Ciencia de Datos

## Módulo 2: Organización y preparación de datos

### Notebook 3: Limpieza y preprocesamiento de datos

En los notebooks anteriores aprendimos a importar, explorar, seleccionar y filtrar datos.

En este notebook abordaremos una etapa fundamental de cualquier análisis: el **control de calidad y limpieza de los datos**.

Los datos reales pueden contener problemas como valores faltantes, fechas ausentes, registros duplicados, tipos de datos incorrectos o valores que no tienen sentido físico.

Al finalizar este notebook podrás:

- identificar valores faltantes;

- detectar fechas ausentes en una serie temporal;

- identificar registros duplicados;

- revisar tipos de datos;

- detectar valores inválidos o sospechosos;

- corregir problemas básicos de calidad;

- preparar una serie temporal limpia para análisis posteriores.

## 1. Importar las librerías

Utilizaremos principalmente `pandas` para organizar, revisar y limpiar los datos.

También importaremos `matplotlib.pyplot`, que utilizaremos posteriormente cuando necesitemos representar gráficamente una serie.

Como en los notebooks anteriores, utilizaremos los nombres abreviados `pd` y `plt`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## 2. Cargar la serie de precipitación

Trabajaremos nuevamente con una serie diaria de precipitación.

Primero importaremos y prepararemos los datos de la misma forma que en los notebooks anteriores.

Ejecuta la siguiente celda y selecciona el archivo:

`BernardoOHigginsChillan.xlsx`

In [ ]:
from google.colab import files

uploaded = files.upload()

## 3. Leer el archivo con Pandas

El archivo contiene una hoja llamada `Serie`.

Utilizaremos la función `read_excel()` de Pandas para importar los datos.

Además de leer el archivo, realizaremos en una misma celda las operaciones de preparación que ya conocemos:

1. construir una variable `fecha`;
2. renombrar la variable de precipitación;
3. conservar solamente las columnas necesarias;
4. utilizar la fecha como índice.

Estas operaciones no constituyen todavía limpieza de datos. Su objetivo es dejar el DataFrame en una estructura adecuada para comenzar el control de calidad.

In [ ]:
datos = pd.read_excel(
    "BernardoOHigginsChillan.xlsx",
    sheet_name="Serie"
)

datos["fecha"] = pd.to_datetime(
    dict(
        year=datos["agno"],
        month=datos["mes"],
        day=datos["dia"]
    )
)

datos = datos.rename(columns={
    "valor": "precipitacion_mm"
})

datos = datos[
    ["fecha", "precipitacion_mm"]
]

datos = datos.set_index("fecha")

datos.head()

El DataFrame debería contener ahora una única variable, `precipitacion_mm`, y utilizar `fecha` como índice.

Esta estructura es especialmente conveniente para trabajar con series temporales, ya que nos permite localizar registros mediante fechas y comprobar directamente la continuidad temporal de la serie.

## 4. Crear una copia de trabajo

Una buena práctica es evitar modificar inmediatamente los datos originales.

Crearemos una copia del DataFrame para realizar las operaciones de limpieza.

La función `copy()` genera un nuevo DataFrame independiente. De esta manera podremos introducir cambios o realizar pruebas sin modificar la variable `datos`.

Conservaremos:

- `datos`: versión original preparada;
- `datos_limpieza`: versión sobre la que realizaremos el control de calidad.

In [ ]:
datos_limpieza = datos.copy()

## 5. Valores faltantes

Un valor faltante ocurre cuando existe una observación en el DataFrame, pero no existe un valor registrado para alguna de sus variables.

En Pandas estos valores suelen representarse como `NaN`.

Por ejemplo:

| fecha | precipitacion_mm |
|---|---:|
| 2025-07-09 | 4.5 |
| 2025-07-10 | NaN |
| 2025-07-11 | 0.0 |

La fecha `2025-07-10` existe, pero no dispone de un valor de precipitación.

Podemos identificar estos casos utilizando `isna()`.

In [ ]:
datos_limpieza.isna().sum()

`isna()` evalúa cada valor del DataFrame y determina si está ausente.

Al combinarlo con `sum()`, contamos cuántos valores faltantes existen en cada columna.

El resultado nos permite responder una primera pregunta de control de calidad:

**¿Existen registros en los que la fecha está presente, pero falta el dato de precipitación?**

Para comprender mejor cómo funciona esta herramienta, introduciremos temporalmente un valor faltante en una copia del DataFrame.

No modificaremos `datos_limpieza`.

En su lugar crearemos `datos_ejemplo` y reemplazaremos deliberadamente el valor de precipitación del `10 de julio de 2025` por `pd.NA`.

`pd.NA` es una representación de Pandas para un dato ausente.

In [ ]:
datos_ejemplo = datos_limpieza.copy()

datos_ejemplo.loc["2025-07-10", "precipitacion_mm"] = pd.NA

Ahora observaremos algunos días antes y después de la fecha modificada.

Como `fecha` es el índice del DataFrame, podemos seleccionar directamente un intervalo mediante `.loc[]`.

Esto permite comprobar visualmente que la fecha sigue existiendo, pero que su valor de precipitación ha desaparecido.

In [ ]:
datos_ejemplo.loc["2025-07-08":"2025-07-12"]

La fila correspondiente al `2025-07-10` continúa presente en el DataFrame.

Lo que falta es solamente el valor de la variable `precipitacion_mm`.

Volvamos a utilizar `isna().sum()` para comprobar si Pandas detecta el cambio.

In [ ]:
datos_ejemplo.isna().sum()

El número de valores faltantes debería haber aumentado en una unidad.

Esto confirma que `isna()` es apropiado para detectar valores ausentes **cuando la fila correspondiente todavía existe**.

Sin embargo, existe otro tipo de problema que `isna()` no puede detectar: una fecha que no aparece en absoluto en el DataFrame.

## 6. Valores faltantes versus fechas faltantes

Un valor `NaN` y una fecha ausente no representan el mismo problema.

### Valor faltante

La fecha existe, pero falta el valor de precipitación.

Por ejemplo:

| fecha | precipitacion_mm |
|---|---:|
| 2025-06-01 | 5.2 |
| 2025-06-02 | NaN |
| 2025-06-03 | 2.1 |

### Fecha faltante

La observación completa no aparece en el DataFrame.

Por ejemplo:

| fecha | precipitacion_mm |
|---|---:|
| 2025-06-01 | 5.2 |
| 2025-06-03 | 2.1 |

En este segundo caso falta completamente el día `2025-06-02`.

Por esta razón, `isna()` no permite detectar fechas que están completamente ausentes.

Para reproducir este segundo problema eliminaremos deliberadamente una fecha de `datos_ejemplo`.

Utilizaremos `drop()` para eliminar completamente el registro correspondiente al `1 de diciembre de 2025`.

Después de esta operación la fecha ya no aparecerá en el índice.

In [ ]:
datos_ejemplo = datos_ejemplo.drop(
    pd.Timestamp("2025-12-01")
)

¿Cómo podemos detectar una fecha que ya no existe?

Primero debemos construir una serie con **todas las fechas que deberían estar presentes**.

Utilizaremos `pd.date_range()`.

Los argumentos indican:

- `start`: primera fecha de nuestra serie;
- `end`: última fecha de nuestra serie;
- `freq="D"`: esperamos una observación cada día.

El resultado será un calendario diario completo entre el inicio y el final del periodo.

In [ ]:
fechas_esperadas = pd.date_range(
    start=datos_ejemplo.index.min(),
    end=datos_ejemplo.index.max(),
    freq="D"
)

fechas_faltantes = fechas_esperadas.difference(
    datos_ejemplo.index
)

print("Número de fechas faltantes:", len(fechas_faltantes))

`difference()` compara el calendario completo generado en `fechas_esperadas` con las fechas realmente disponibles en el índice de `datos_ejemplo`.

Todas las fechas que deberían existir pero no aparecen en el DataFrame quedan almacenadas en `fechas_faltantes`.

Esta comprobación es diferente de `isna()`:

- `isna()` detecta valores ausentes dentro de filas existentes;
- `difference()` permite detectar días completos que no aparecen en la serie.

Podemos mostrar individualmente cada fecha faltante utilizando un ciclo `for`.

La función `strftime()` se utiliza únicamente para mostrar las fechas con el formato:

`año-mes-día`

In [ ]:
for fecha in fechas_faltantes:
    print(fecha.strftime("%Y-%m-%d"))

Al revisar una serie temporal real conviene comprobar **ambos tipos de ausencia**:

1. valores `NaN`;
2. fechas completamente ausentes.

Una serie podría no contener ningún `NaN` y, aun así, presentar interrupciones temporales.

## 7. Registros duplicados

Otra situación frecuente ocurre cuando una misma observación aparece más de una vez.

En una serie diaria de precipitación esperamos disponer de **una sola observación por fecha**.

Por ejemplo, si el `15 de junio de 2025` aparece dos veces, debemos revisar cuál es la causa de la duplicación antes de utilizar esos datos.

Como estamos utilizando la fecha como índice, podemos comprobar directamente si el índice contiene valores repetidos mediante `duplicated()`.

In [ ]:
datos_ejemplo.index.duplicated().sum()

`duplicated()` identifica las fechas repetidas y devuelve valores `True` o `False`.

Al utilizar `sum()` contamos cuántas fechas están duplicadas.

Para observar cómo funciona, introduciremos deliberadamente un registro repetido.

Primero seleccionaremos la fila correspondiente al `15 de junio de 2025`.

In [ ]:
fila_duplicada = datos_ejemplo.loc[["2025-06-15"]]

datos_con_duplicado = pd.concat(
    [datos_ejemplo, fila_duplicada]
)

datos_con_duplicado.index.duplicated().sum()

`pd.concat()` permite unir DataFrames.

En este ejemplo añadimos nuevamente una fila que ya estaba presente. Por lo tanto, ahora existen dos registros con la misma fecha.

El siguiente paso consiste en localizar todas las filas involucradas en la duplicación.

Utilizaremos:

`keep=False`

para indicar que queremos marcar **todas las apariciones** de una fecha duplicada, no solamente una de ellas.

In [ ]:
datos_con_duplicado[
    datos_con_duplicado.index.duplicated(keep=False)
]



Ahora podemos observar directamente los registros repetidos.

Antes de eliminar duplicados es importante revisarlos.

Dos registros con la misma fecha podrían:

- contener exactamente el mismo valor;
- contener valores diferentes;
- corresponder a un problema de importación;
- representar información que requiere una revisión adicional.

Cuando decidimos conservar solamente una aparición podemos utilizar `keep="first"`.

Esto indica que se conserva la primera observación encontrada y se elimina la repetición posterior.

In [ ]:
datos_sin_duplicados = datos_con_duplicado[
    ~datos_con_duplicado.index.duplicated(keep="first")
]

El símbolo `~` significa negación lógica.

En este caso:

`datos_con_duplicado.index.duplicated(...)`

identifica las observaciones duplicadas.

Al colocar `~` delante, seleccionamos aquellas filas que **no deben ser eliminadas**.

El resultado se almacena en un nuevo DataFrame llamado `datos_sin_duplicados`.

## 8. Revisar los tipos de datos

Una variable puede contener valores aparentemente correctos, pero estar almacenada con un tipo incorrecto.

Por ejemplo, una columna de precipitación debería contener valores numéricos.

Si una columna contiene una mezcla de números y texto, Pandas puede interpretarla como `object`. En ese caso algunas operaciones matemáticas podrían no funcionar correctamente.

Podemos revisar los tipos mediante `dtypes`.

In [ ]:
datos_limpieza.dtypes

En condiciones normales esperamos que `precipitacion_mm` tenga un tipo numérico, normalmente `float64`.

Para observar qué ocurre cuando aparece contenido no numérico, crearemos nuevamente una copia de trabajo e introduciremos deliberadamente el texto:

`"error"`

en una de las observaciones.

Este ejemplo representa una situación que puede aparecer en archivos reales cuando una columna contiene códigos, texto, símbolos o registros mal ingresados.

In [ ]:
datos_tipo = datos_limpieza.copy()

datos_tipo.loc["2025-06-20", "precipitacion_mm"] = "error"

Una columna estrictamente numérica no está diseñada para almacenar texto.

Para reproducir de forma explícita una columna con contenido mixto, cambiaremos temporalmente su tipo a `object`.

Posteriormente volveremos a introducir `"error"` dentro de la columna.

In [ ]:
datos_tipo["precipitacion_mm"] = datos_tipo["precipitacion_mm"].astype("object")
datos_tipo.loc["2025-06-20", "precipitacion_mm"] = "error"

Revisemos ahora el tipo de dato.

Si la columna aparece como `object`, Pandas ya no la está interpretando como una variable puramente numérica.

In [ ]:
datos_tipo.dtypes

El siguiente paso consiste en intentar recuperar una columna numérica.

Para ello utilizaremos:

`pd.to_numeric()`

Esta función intenta convertir cada valor de la columna a un número.

### ¿Qué significa `errors="coerce"`?

Supongamos que una columna contiene:

`5.2`

`8.4`

`error`

`2.1`

Los tres valores numéricos pueden convertirse sin dificultad, pero `"error"` no puede interpretarse como un número.

Al utilizar:

`errors="coerce"`

Pandas convierte los valores válidos y transforma aquellos que no puede convertir en `NaN`.

El resultado sería:

`5.2`

`8.4`

`NaN`

`2.1`

Esto es útil durante la limpieza porque permite identificar los registros problemáticos sin detener la ejecución del código.

**Importante:** convertir un valor incorrecto en `NaN` no significa que el problema haya sido solucionado. Significa que hemos logrado identificarlo como un dato que requiere revisión.

In [ ]:
datos_tipo["precipitacion_mm"] = pd.to_numeric(
    datos_tipo["precipitacion_mm"],
    errors="coerce"
)

Después de la conversión debemos comprobar si aparecieron valores faltantes.

Si el número de `NaN` aumenta después de utilizar `pd.to_numeric()`, significa que uno o más registros no pudieron convertirse correctamente a números.

In [ ]:
datos_tipo.isna().sum()

En nuestro ejemplo esperamos encontrar el `NaN` correspondiente al registro donde introdujimos deliberadamente `"error"`.

En una base de datos real este procedimiento ayuda a localizar valores que parecían formar parte de una columna numérica, pero que en realidad contenían información incompatible.

## 9. Valores inválidos o sospechosos

La limpieza de datos no consiste solamente en encontrar `NaN`.

También debemos comprobar si los valores tienen sentido para la variable que estamos analizando.

En el caso de la precipitación:

- valores negativos no tienen sentido físico;
- valores extremadamente grandes pueden ser reales o pueden corresponder a errores y deben revisarse.

Existe una diferencia importante entre un valor **inválido** y un valor **extremo**.

Un valor negativo de precipitación es físicamente incompatible con la variable.

En cambio, una precipitación diaria muy alta podría corresponder a un evento meteorológico extremo real.

No debemos eliminar automáticamente un valor simplemente porque sea grande.

Para demostrar la detección de valores inválidos introduciremos deliberadamente una precipitación de `-15 mm`.

Utilizaremos nuevamente una copia para no modificar nuestros datos originales.

In [ ]:
datos_error = datos_limpieza.copy()

datos_error.loc["2025-07-15", "precipitacion_mm"] = -15

Ahora utilizaremos una condición lógica para localizar todas las observaciones cuya precipitación sea menor que cero.

La expresión:

`datos_error["precipitacion_mm"] < 0`

produce valores `True` para los registros que cumplen esa condición y `False` para los demás.

Utilizamos esa condición para filtrar el DataFrame.

In [ ]:
datos_error[
    datos_error["precipitacion_mm"] < 0
]

El registro introducido artificialmente debería aparecer en el resultado.

En datos reales, encontrar un valor negativo de precipitación indicaría que ese registro requiere revisión antes de continuar con el análisis.

Ahora volveremos al DataFrame original de limpieza para revisar el rango real de los datos.

Comenzaremos obteniendo la precipitación mínima mediante `min()`.

In [ ]:
datos_limpieza["precipitacion_mm"].min()

Para una serie de precipitación esperamos que el valor mínimo sea igual o superior a cero.

A continuación revisaremos el valor máximo de toda la serie.

In [ ]:
datos_limpieza["precipitacion_mm"].max()

Un valor máximo elevado no debe considerarse automáticamente un error.

Para evaluar mejor los eventos extremos conviene observar varios de los valores más altos, en lugar de revisar únicamente el máximo absoluto.

La función `nlargest()` permite obtener las observaciones con los mayores valores de una columna.

Mostraremos los diez días con mayor precipitación.

In [ ]:
datos_limpieza.nlargest(
    10,
    "precipitacion_mm"
)

Los valores altos no deben considerarse automáticamente errores.

Un valor extremo puede corresponder a un evento meteorológico real. Su validez debe evaluarse utilizando conocimiento del fenómeno, información de estaciones cercanas u otras fuentes de datos.

La revisión de estos registros permite preguntarnos:

- ¿el máximo está aislado?
- ¿existen otros eventos de magnitud similar?
- ¿el valor es razonable para la estación y su contexto climático?

El control de calidad requiere combinar herramientas computacionales con conocimiento de la variable que estamos analizando.

## 10. Verificar el orden temporal

Antes de trabajar con una serie temporal debemos comprobar que las observaciones estén ordenadas cronológicamente.

Al utilizar la fecha como índice podemos verificar directamente si las fechas avanzan desde la más antigua hacia la más reciente.

La propiedad:

`is_monotonic_increasing`

devuelve:

- `True` si el índice está ordenado de menor a mayor;
- `False` si existen fechas fuera de orden.

In [ ]:
datos_limpieza.index.is_monotonic_increasing

Aunque el resultado sea `True`, podemos utilizar `sort_index()` para asegurarnos de que la serie quede ordenada cronológicamente según su índice.

Esta operación será especialmente importante cuando realicemos posteriormente:

- filtros temporales;
- gráficos;
- agregaciones diarias, mensuales o anuales;
- otros análisis de series temporales.

In [ ]:
datos_limpieza = datos_limpieza.sort_index()

## 11. Evaluar la completitud de la serie

Además de identificar fechas faltantes, resulta útil resumir qué proporción del periodo esperado está realmente disponible.

La **completitud temporal** compara:

- el número de fechas únicas presentes en el DataFrame;
- el número total de fechas que deberían existir entre el inicio y el final de la serie.

Primero construiremos nuevamente el calendario diario completo con `pd.date_range()`.

In [ ]:
fechas_esperadas = pd.date_range(
    datos_limpieza.index.min(),
    datos_limpieza.index.max(),
    freq="D"
)

completitud = (
    len(datos_limpieza.index.unique())
    / len(fechas_esperadas)
) * 100

print(f"Completitud temporal: {completitud:.2f}%")

El resultado se expresa como porcentaje.

Por ejemplo, una completitud temporal de `98 %` significa que aproximadamente el 98 % de las fechas esperadas está presente en el DataFrame.

Sin embargo, este cálculo solamente evalúa si la **fecha existe**.

Una fecha puede aparecer en el DataFrame y contener un valor `NaN`.

Por esta razón debemos calcular también la **completitud de los datos**, considerando únicamente las observaciones que tienen un valor de precipitación válido.

In [ ]:
n_validos = datos_limpieza["precipitacion_mm"].notna().sum()

completitud_datos = (
    n_validos / len(fechas_esperadas)
) * 100

print(f"Completitud de datos: {completitud_datos:.2f}%")

`notna()` identifica las observaciones que sí contienen un valor válido.

Por lo tanto, estos dos indicadores responden preguntas diferentes:

### Completitud temporal

**¿Qué porcentaje de las fechas esperadas existe en el DataFrame?**

### Completitud de datos

**¿Qué porcentaje de las observaciones esperadas contiene realmente un valor válido de precipitación?**

Esta diferencia vuelve a mostrar por qué una fecha faltante y un valor `NaN` no representan exactamente el mismo problema.

## 12. Resumen del control de calidad

Después de revisar individualmente diferentes problemas, podemos reunir algunas comprobaciones básicas para obtener una visión general del estado de nuestra serie.

El siguiente bloque mostrará:

- número total de registros;
- cantidad de valores faltantes;
- número de fechas duplicadas;
- cantidad de precipitaciones negativas;
- fecha inicial;
- fecha final.

Este tipo de resumen permite realizar una revisión rápida antes de continuar con el análisis.

In [ ]:
print("Registros:", len(datos_limpieza))
print("Valores faltantes:", datos_limpieza["precipitacion_mm"].isna().sum())
print("Fechas duplicadas:", datos_limpieza.index.duplicated().sum())
print("Valores negativos:", (datos_limpieza["precipitacion_mm"] < 0).sum())
print("Fecha inicial:", datos_limpieza.index.min())
print("Fecha final:", datos_limpieza.index.max())

Este resumen no reemplaza las comprobaciones anteriores.

Su utilidad es reunir varios indicadores en un solo lugar para responder rápidamente:

- ¿cuántos datos tenemos?
- ¿existen valores faltantes?
- ¿hay fechas duplicadas?
- ¿aparecen valores físicamente inválidos?
- ¿qué periodo cubre la serie?

Una base de datos debería pasar por este tipo de revisión antes de utilizarse en análisis estadísticos, visualizaciones o modelos.

## Tarea

Preparación y control de calidad de una serie de temperatura

En esta actividad aplicarán de manera autónoma los contenidos trabajados hasta ahora en el curso. Para resolverla deberán reutilizar y adaptar código de los notebooks anteriores.

No se entregará código nuevo para esta actividad. El objetivo es aprender a identificar qué herramientas ya utilizadas pueden aplicarse a un nuevo conjunto de datos.

Objetivo

Construir, revisar y analizar una serie diaria de temperatura obtenida desde el Explorador Climático CR2, aplicando las herramientas de Python y Pandas aprendidas durante el curso.

1. Obtención de los datos

Ingresa al Explorador Climático CR2 y selecciona una estación ubicada en la Región de Ñuble que disponga de registros diarios de temperatura.

Descarga la serie disponible en formato Excel.

Registra en una celda Markdown:

* nombre de la estación;
* variable descargada;
* ubicación;
* periodo disponible en la serie.

2. Importación de los datos

Utilizando Google Colab:

1. Carga el archivo descargado.
2. Importa las librerías necesarias.
3. Lee la hoja que contiene la serie utilizando Pandas.
4. Guarda los datos en un DataFrame.
5. Muestra las primeras y últimas observaciones.

3. Exploración inicial

Utilizando las herramientas aprendidas, determina:

* número de filas y columnas;
* nombres de las variables;
* tipos de datos;
* fecha inicial;
* fecha final.

Utiliza, cuando corresponda:

head(), tail(), shape, columns, dtypes e info().

Incluye una breve explicación de lo que observas.

4. Preparación del DataFrame

Construye una única variable fecha a partir de las columnas correspondientes a año, mes y día.

Posteriormente:

1. Convierte fecha al tipo datetime.
2. Utiliza fecha como índice del DataFrame.
3. Renombra la variable de temperatura como:

temperatura_C

4. Elimina las columnas que ya no sean necesarias.

El DataFrame final debe contener solamente:

* fecha como índice;
* temperatura_C como única columna.

El resultado esperado debe tener una estructura similar a:

fecha	temperatura_C
2025-01-01	18.4
2025-01-02	19.1
2025-01-03	17.8

5. Control de calidad

Realiza un control básico de calidad de la serie.

Determina:

1. Número de valores faltantes (NaN).
2. Número de fechas faltantes en la serie diaria.
3. Lista completa de fechas faltantes.
4. Número de fechas duplicadas.
5. Tipo de dato de temperatura_C.
6. Temperatura mínima de toda la serie.
7. Temperatura máxima de toda la serie.

Revisa además si existen valores que consideres sospechosos o poco razonables.

Importante: un valor extremo no debe eliminarse automáticamente. Primero debe identificarse y evaluarse.

6. Selección y filtrado

Utilizando las herramientas de selección aprendidas:

1. Selecciona únicamente los datos correspondientes a un año completo desde 2020, cualquier año.
2. Selecciona el invierno meteorológico de ese año:
    * junio;
    * julio;
    * agosto.
3. Determina:
    * temperatura mínima durante JJA dde ese año;
    * fecha en que ocurrió;
    * temperatura máxima durante JJA de ese año;
    * fecha en que ocurrió.
4. Identifica los 10 días más fríos de JJA dde ese año.

7. Visualización

Realiza un gráfico de la temperatura diaria durante junio, julio y agosto de ese año.

El gráfico debe incluir como mínimo:

* fecha en el eje x;
* temperatura en °C en el eje y;
* título;
* nombre de la estación;
* unidades correctamente indicadas.

8. Interpretación

En una celda Markdown responde brevemente:

1. ¿La serie presenta datos o fechas faltantes?
2. ¿Observaste algún valor que consideres sospechoso? ¿Por qué?
3. ¿Cuál fue el día más frío del invierno de ese año?
4. ¿Cuál fue el día más cálido del invierno de ese año?
5. ¿Qué características generales observas en la evolución de la temperatura durante JJA de ese año?

9. Resultado final

Antes de terminar, verifica que tu notebook contenga:

* información de la estación seleccionada;
* importación de los datos;
* exploración inicial;
* DataFrame con fecha como índice;
* únicamente temperatura_C como columna;
* control de valores faltantes;
* control de fechas faltantes;
* control de duplicados;
* temperatura mínima y máxima;
* análisis de JJA 2025;
* gráfico de temperatura de JJA de ese año;
* respuestas a las preguntas de interpretación.

Entrega

Entrega el notebook de Google Colab completo, manteniendo tanto las celdas de código como las explicaciones en celdas Markdown.

El notebook debe poder ejecutarse desde el inicio hasta el final y producir los resultados presentados.

Recomendación

No escribas todo el código desde cero. Revisa los notebooks anteriores e identifica qué fragmentos puedes reutilizar y adaptar para trabajar ahora con temperatura en lugar de precipitación.

Parte importante del trabajo consiste precisamente en aprender a reconocer cuándo una solución desarrollada anteriormente puede aplicarse a un problema nuevo.